# Create protein mutants

Loads a wild-type protein, applies point mutations (e.g. `D100A`) to its sequence and saves the mutant as a new protein JSON.

In [ ]:
import re
from chromhandler import Protein


def create_mutant(wt: Protein, mutations: list[str], name: str | None = None) -> Protein:
    """Returns a copy of `wt` with point mutations like 'D100A' applied (1-based positions).

    The wild-type residue is checked against the sequence, so a wrong position raises an error.
    """
    if wt.sequence is None:
        raise ValueError(f"{wt.name} has no sequence, refetch it with Protein.from_uniprot()")

    seq = list(wt.sequence)
    for mutation in mutations:
        match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", mutation)
        if match is None:
            raise ValueError(f"Invalid mutation '{mutation}', expected e.g. 'D100A'")
        wt_aa, pos, new_aa = match.group(1), int(match.group(2)), match.group(3)
        if not 1 <= pos <= len(seq):
            raise ValueError(f"Position {pos} is outside the sequence (length {len(seq)})")
        if seq[pos - 1] != wt_aa:
            raise ValueError(f"{mutation}: residue {pos} is {seq[pos - 1]}, not {wt_aa}")
        seq[pos - 1] = new_aa

    suffix = "_".join(mutations)
    return Protein(
        id=f"{wt.id}_{suffix}",
        name=name or f"{wt.name}_{suffix}",
        sequence="".join(seq),
        organism=wt.organism,
        organism_tax_id=wt.organism_tax_id,
        init_conc=wt.init_conc,
        conc_unit=wt.conc_unit,
        constant=wt.constant,
    )

In [ ]:
wt = Protein.read_json("result/proteins/EbPPK.json")

# mutations as <wild-type residue><position><new residue>, positions are 1-based
mutant = create_mutant(wt, ["P100A"])

path = f"result/proteins/{mutant.name}.json"
mutant.save_json(path)
print(f"Saved {mutant.name} ({mutant.id}) to {path}")